# XOR Problem

In [1]:
from neuron import *
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from functools import partial

In [2]:
def i_fn(t, i_max, t_start, t_end):
    if t < t_start or t > t_end:
        return 0
    return i_max

i_fn_map = {
    0: partial(i_fn, i_max=-5, t_start=0, t_end=10),
    1: partial(i_fn, i_max=5, t_start=0, t_end=10),
}

In [14]:
x1 = 1
x2 = 1

x1_injector = CurrentInjector(i_fn_map[x1])
x2_injector = CurrentInjector(i_fn_map[x2])
neuron1 = Neuron(l=1000, d=10.0, r_a=25.0)
neuron2 = Neuron(l=1000, d=10.0, r_a=25.0)
synapse11 = Synapse(x1_injector, neuron1, delay=0.5, weight=-1.0)
synapse12 = Synapse(x1_injector, neuron2, delay=0.5, weight=1.0)
synapse21 = Synapse(x2_injector, neuron1, delay=0.5, weight=1.0)
synapse22 = Synapse(x2_injector, neuron2, delay=0.5, weight=-1.0)

neuron_end = Neuron(l=1000, d=10.0)
synapse1_end = Synapse(neuron1, neuron_end, g=0.4, e_syn=-65.0, delay=0.5)
synapse2_end = Synapse(neuron2, neuron_end, g=0.4, e_syn=-65.0, delay=0.5)

recorder = Recorder(100, neuron1, neuron2, neuron_end)
recorder.run_network(synapse11, synapse12, synapse21, synapse22, synapse1_end, synapse2_end)

100%|██████████| 10000/10000 [00:01<00:00, 5402.37it/s]


In [15]:
data = []
neurons = [neuron1, neuron2, neuron_end]
for i in range(len(neurons)):
    neuron = neurons[i]
    for t in range(0, recorder.n_t, 200):
        for x in range(neuron.n_x):
            data.append({"neuron": i, "time": t, "x": neuron.dx * x, "v": recorder.v[i][x, t]})
df = pd.DataFrame(data)

In [16]:
fig = px.line(df, x="x", y="v", animation_frame="time", title="Voltage of Neurons", facet_row="neuron", range_y=[-100, 40])
fig.update_layout(height=1000)
fig.show()